In [115]:
import pandas as pd
import re
import numpy as np

src = "../data/gastat_foreign_trade.csv"
df = pd.read_csv(src)

df.head()

,Country ID,Country,Year,Section ID,Section,Trade Flow ID,Trade Flow,Million SAR
0,abw,Aruba,2022,Animal and Vegetable Fats; Oils; Waxes and The...,NaN,1,Imports,0.000000
1,abw,Aruba,2022,"Articles of Stone, of Plaster, Cement, Asbesto...",NaN,1,Imports,0.011998
2,abw,Aruba,2022,Base Metals and Articles of Base Metals,NaN,1,Imports,0.198855
3,abw,Aruba,2022,"Footwear, Headgear, Umbrellas, Whips, Artifici...",NaN,1,Imports,0.000000
4,abw,Aruba,2022,Live Animals; Animal Products,NaN,1,Imports,0.000000


In [116]:
code_map = (
    df.loc[df["Section ID"].astype(str).str.match(r"^S\d{1,2}$", na=False) & df["Section"].notna(), ["Section ID", "Section"]]
    .assign(
        Section_ID=lambda x: x["Section ID"].astype(str).str.strip().str.upper(),
        Section_Name=lambda x: x["Section"].astype(str).str.strip()
    )
    .groupby("Section_ID")["Section_Name"]
    .agg(lambda s: s.value_counts().index[0])
    .to_dict()
)

code_map

{'S01': 'Animals and their products',
 'S02': 'Plant Products',
 'S03': 'Animal and vegetable fats, oils, waxex and their products',
 'S04': 'Prepared foodstuffs, beverages and tobacco',
 'S05': 'Mineral Products',
 'S06': 'Products of the chemical industries',
 'S07': 'Plastics, rubber and their articles',
 'S08': 'Skins, leather and their articles, handbags and similar',
 'S09': 'Wood, cork, plaiting materials and their articles',
 'S10': 'Paper, paperboard and their articles',
 'S11': 'Textiles and their articles',
 'S12': 'Footwear, headgear, umbrellas, sticks',
 'S13': 'Articles of stone, plaster, cement, ceramic, glass',
 'S14': 'Precious stones or metals and their articles, jewelry',
 'S15': 'Base metals and their articles',
 'S16': 'Machinery and mechanical appliances, electrical equipment, parts thereof',
 'S17': 'Vehicles, aircraft, vessels and associated transport equipment',
 'S18': 'Optical, photographic, measuring, checking, medical instruments and apparatus; clocks and m

In [117]:
long_to_code = {
    "Animal and Vegetable Fats; Oils; Waxes and Their Products": "S03",
    "Articles of Stone, of Plaster, Cement, Asbestos, Mica, Ceramic Products, Glassware": "S13",
    "Base Metals and Articles of Base Metals": "S15",
    "Footwear, Headgear, Umbrellas, Whips, Artificial Flowers, Articles of Human Hair": "S12",
    "Live Animals; Animal Products": "S01",
    "Machinery and Mechanical Appliances; Electrical Equipment; Parts Thereof": "S16",
    "Optical, Photographic, Cinematographic, Clocks and Watches Parts Thereof": "S18",
    "Paper-Making Material; Paper and Articles Thereof": "S10",
    "Pearls, Precious and Semi-Precious, False Jewelry": "S14",
    "Plastics and Articles Thereof; Rubber and Articles Thereof": "S07",
    "Prepared Foodstuffs; Beverages; Tobacco": "S04",
    "Products of The Chemical and Allied Industries": "S06",
    "Raw Hides and Skins, Leather, Furs and Articles Thereof; Suitcases; Handbags and Similar": "S08",
    "Textiles and Textile Articles": "S11",
    "Transport Equipment and Parts Thereof": "S17",
    "Vegetable Products": "S02",
    "Wood and Articles of Wood; Wood Charcoal; Cork and Articles of Cork": "S09",
    "Works of Arts, Collectors Pieces and Antiques": "S21"
}

In [118]:
def section_code(row):
    sid = str(row["Section ID"]).strip() if pd.notna(row["Section ID"]) else ""
    sid_up = sid.upper()
    if re.fullmatch(r"S\d{1,2}", sid_up):
        if len(sid_up) == 2:
            sid_up = "S0" + sid_up[1]
        return sid_up
    return long_to_code.get(sid, None)

def canonical_section(row):
    code = section_code(row)
    if code and code in code_map:
        return code_map[code]
    sec = str(row["Section"]).strip() if pd.notna(row["Section"]) else ""
    return sec

In [119]:
df2 = df.copy()

for c in ["Section ID", "Section", "Country ID", "Country", "Trade Flow" , "Trade Flow"]:
    df2[c] = df2[c].astype(str).replace("nan", np.nan).str.strip()

df2["Section_Code"] = df2.apply(section_code, axis=1)
df2["Section_21"] = df2.apply(canonical_section, axis=1).astype(str).str.strip()

df2[["Section ID", "Section", "Section_Code", "Section_21"]].head(10)

,Section ID,Section,Section_Code,Section_21
0,Animal and Vegetable Fats; Oils; Waxes and The...,NaN,S03,"Animal and vegetable fats, oils, waxex and the..."
1,"Articles of Stone, of Plaster, Cement, Asbesto...",NaN,S13,"Articles of stone, plaster, cement, ceramic, g..."
2,Base Metals and Articles of Base Metals,NaN,S15,Base metals and their articles
3,"Footwear, Headgear, Umbrellas, Whips, Artifici...",NaN,S12,"Footwear, headgear, umbrellas, sticks"
4,Live Animals; Animal Products,NaN,S01,Animals and their products
5,Machinery and Mechanical Appliances; Electrica...,NaN,S16,"Machinery and mechanical appliances, electrica..."
6,"Optical, Photographic, Cinematographic, Clocks...",NaN,S18,"Optical, photographic, measuring, checking, me..."
7,Paper-Making Material; Paper and Articles Thereof,NaN,S10,"Paper, paperboard and their articles"
8,"Pearls, Precious and Semi-Precious, False Jewelry",NaN,S14,"Precious stones or metals and their articles, ..."
9,Plastics and Articles Thereof; Rubber and Arti...,NaN,S07,"Plastics, rubber and their articles"


In [120]:
df2["Trade Flow"] = df2["Trade Flow"].astype(str).replace("nan", np.nan).str.strip().str.title()
df2["Trade Flow"] = df2["Trade Flow"].replace({"Import": "Imports", "Export": "Exports"})

df2["Year"] = pd.to_numeric(df2["Year"], errors="coerce").astype("Int64")
df2["Million SAR"] = pd.to_numeric(df2["Million SAR"], errors="coerce")

df2 = df2.dropna(subset=["Country", "Year", "Trade Flow", "Million SAR", "Section_21", "Section_Code"])
df2 = df2[df2["Million SAR"] >= 0]
df2 = df2[df2["Year"].between(2022, 2024)]

In [121]:
df_clean21 = df2[["Country ID", "Country", "Year", "Section_Code", "Section_21","Trade Flow ID","Trade Flow", "Million SAR"]].rename(
    columns={"Section_Code": "Section ID", "Section_21": "Section"}
)

df_clean21.to_csv("../data/gastat_foreign_trade_cleaned.xlsx", index=False)

df_clean21.head()

,Country ID,Country,Year,Section ID,Section,Trade Flow ID,Trade Flow,Million SAR
0,abw,Aruba,2022,S03,"Animal and vegetable fats, oils, waxex and the...",1,Imports,0.000000
1,abw,Aruba,2022,S13,"Articles of stone, plaster, cement, ceramic, g...",1,Imports,0.011998
2,abw,Aruba,2022,S15,Base metals and their articles,1,Imports,0.198855
3,abw,Aruba,2022,S12,"Footwear, headgear, umbrellas, sticks",1,Imports,0.000000
4,abw,Aruba,2022,S01,Animals and their products,1,Imports,0.000000


In [122]:
df_clean21.shape

(25107, 8)